# Geospatial Data Retrieval GUI

This notebook provides a simple graphical interface for the `geospatial_data_retrieval_agent`.

Users type a natural-language data request, click **Download and Map**, and the returned vector dataset is displayed in an interactive map.


In [ ]:
import json
import sys
from pathlib import Path
from urllib.parse import urljoin

import geopandas as gpd
import ipywidgets as widgets
import requests
from IPython.display import HTML, display

try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except Exception:
    pass

try:
    import folium
except ImportError as exc:
    raise ImportError("This notebook needs folium. Install it with: pip install folium") from exc

project_root = Path.cwd()
if project_root.name == "examples":
    project_root = project_root.parent

examples_dir = project_root / "examples"
artifact_dir = examples_dir / "downloaded_gas_artifacts"
artifact_dir.mkdir(parents=True, exist_ok=True)

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from gas_client import GasClient


## Basic Settings

Edit these values if your GAS server or API key is different. Source-specific keys are entered in the GUI below as JSON.


In [ ]:
server_url = "http://127.0.0.1:4042"
openai_api_key = "YOUR_OPENAI_API_KEY"

client = GasClient(server_url, openai_api_key=openai_api_key)
retrieval_agent = client.agent("geospatial_data_retrieval_agent")


## Helper Functions

In [ ]:
download_dir = artifact_dir

last_result = None
last_artifact_paths = []
last_map = None


def absolute_url(url):
    if url.startswith("/"):
        return urljoin(server_url, url)
    return url


def artifact_name(artifact, index):
    name = artifact.get("filename") or artifact.get("name") or f"artifact_{index}"
    return Path(str(name).split("?")[0]).name


def download_artifact(artifact, index):
    url = artifact.get("url")
    if not url:
        return None

    response = requests.get(absolute_url(url), timeout=180)
    response.raise_for_status()

    local_path = download_dir / artifact_name(artifact, index)
    local_path.write_bytes(response.content)
    return local_path


def load_vector(path):
    suffix = path.suffix.lower()
    if suffix in {".geojson", ".json", ".gpkg", ".shp"}:
        return gpd.read_file(path)
    return None


def make_interactive_map(gdf, layer_name="Downloaded data"):
    if gdf.empty:
        raise ValueError("The returned vector dataset has no features.")

    if gdf.crs is not None:
        map_gdf = gdf.to_crs(epsg=4326)
    else:
        map_gdf = gdf.set_crs(epsg=4326, allow_override=True)

    bounds = map_gdf.total_bounds
    center_lat = float((bounds[1] + bounds[3]) / 2)
    center_lon = float((bounds[0] + bounds[2]) / 2)

    m = folium.Map(location=[center_lat, center_lon], zoom_start=8, tiles="CartoDB positron")

    popup_fields = [column for column in map_gdf.columns if column != map_gdf.geometry.name][:8]
    tooltip = None
    if popup_fields:
        tooltip = folium.GeoJsonTooltip(fields=popup_fields, aliases=popup_fields, sticky=False)

    folium.GeoJson(
        data=json.loads(map_gdf.to_json()),
        name=layer_name,
        tooltip=tooltip,
        style_function=lambda feature: {
            "color": "#2563eb",
            "weight": 2,
            "fillColor": "#60a5fa",
            "fillOpacity": 0.35,
        },
        marker=folium.CircleMarker(radius=5, fill=True),
    ).add_to(m)

    m.fit_bounds([[float(bounds[1]), float(bounds[0])], [float(bounds[3]), float(bounds[2])]])
    folium.LayerControl().add_to(m)
    return m


def parse_source_credentials(text_or_dict):
    if isinstance(text_or_dict, dict):
        return text_or_dict
    text = str(text_or_dict or "").strip()
    if not text:
        return {}
    value = json.loads(text)
    if not isinstance(value, dict):
        raise ValueError("Source credentials must be a JSON object.")
    return value


def run_retrieval_request(task_text, source_credentials=None, artifact_delivery="URL", show_progress=True):
    global last_result, last_artifact_paths, last_map

    last_artifact_paths = []
    source_credentials = parse_source_credentials(source_credentials)
    credentials = {}
    if source_credentials:
        credentials["source_credentials"] = source_credentials

    result = None
    for event in retrieval_agent.execute_task(
        task_text,
        mode="stream",
        artifact_delivery=artifact_delivery,
        credentials=credentials,
        timeout=2400,
    ):
        if show_progress:
            client.print_stream_event(event)
        if event.get("event") == "task_result":
            result = event.get("payload")

    if result is None:
        raise RuntimeError("The stream ended before returning a task_result event.")

    last_result = result
    client.print_task_summary(result)

    artifacts = result.get("outputs", {}).get("artifacts", [])
    vector_gdf = None
    vector_path = None

    for index, artifact in enumerate(artifacts, start=1):
        path = download_artifact(artifact, index)
        if path is None:
            continue
        last_artifact_paths.append(path)
        loaded = load_vector(path)
        if loaded is not None:
            vector_gdf = loaded
            vector_path = path
            break

    if vector_gdf is None:
        print("No returned artifact was directly readable as a vector dataset for mapping.")
        return result

    print("Mapping:", vector_path)
    print("Features:", len(vector_gdf))
    print("Columns:", list(vector_gdf.columns))
    last_map = make_interactive_map(vector_gdf, layer_name=vector_path.name)
    display(last_map)
    return result


## GUI

In [ ]:
def show_retrieval_gui():
    request_box = widgets.Textarea(
        value="Download PA county boundaries as GeoJSON.",
        placeholder="Describe the data you want to download...",
        description="Request",
        layout=widgets.Layout(width="100%", height="110px"),
        style={"description_width": "90px"},
    )

    source_credentials_box = widgets.Textarea(
        value=json.dumps(
            {
                "OpenTopography": {"key": "3a309d56a2428f989806a7f52226e085"},
                "US_Census_demography": {"key": "918876e93cf30566c02b367fcad29644d861817e"},
            },
            indent=2,
        ),
        placeholder='{"OpenTopography": {"key": "..."}}',
        description="Keys JSON",
        layout=widgets.Layout(width="100%", height="130px"),
        style={"description_width": "90px"},
    )

    artifact_delivery_dropdown = widgets.Dropdown(
        options=["URL", "Encoded"],
        value="URL",
        description="Delivery",
        style={"description_width": "90px"},
    )

    run_button = widgets.Button(
        description="Download and Map",
        button_style="primary",
        icon="download",
    )

    clear_button = widgets.Button(
        description="Clear Output",
        icon="trash",
    )

    progress_output = widgets.Output()
    map_output = widgets.Output()

    def on_clear_clicked(_button):
        progress_output.clear_output()
        map_output.clear_output()

    def on_run_clicked(_button):
        progress_output.clear_output()
        map_output.clear_output()

        with progress_output:
            try:
                task_text = request_box.value.strip()
                if not task_text:
                    raise ValueError("Please enter a data request.")

                result = run_retrieval_request(
                    task_text,
                    source_credentials=parse_source_credentials(source_credentials_box.value),
                    artifact_delivery=artifact_delivery_dropdown.value,
                    show_progress=True,
                )
                if last_map is not None:
                    with map_output:
                        display(last_map)
            except Exception as exc:
                print("Error:", exc)

    run_button.on_click(on_run_clicked)
    clear_button.on_click(on_clear_clicked)

    controls = widgets.VBox(
        [
            request_box,
            source_credentials_box,
            artifact_delivery_dropdown,
            widgets.HBox([run_button, clear_button]),
            widgets.HTML("<hr>"),
            widgets.HTML("<b>Progress and artifacts</b>"),
            progress_output,
            widgets.HTML("<hr>"),
            widgets.HTML("<b>Interactive map</b>"),
            map_output,
        ]
    )
    display(controls)
    return controls


controls = show_retrieval_gui()


## Non-Widget Fallback

If the GUI does not render in your notebook frontend, run this cell instead. It uses normal Python variables and still displays the interactive Folium map.


In [ ]:
task_text = "Download PA county boundaries as GeoJSON."
source_credentials = {
    "OpenTopography": {"key": "3a309d56a2428f989806a7f52226e085"},
    "US_Census_demography": {"key": "918876e93cf30566c02b367fcad29644d861817e"},
}

# Uncomment to run without the widget UI:
# run_retrieval_request(task_text, source_credentials=source_credentials, artifact_delivery="URL")


## Programmatic Access to the Last Result

After using the GUI, these variables are available for follow-up work:

- `last_result`: the full GAS task result dictionary
- `last_artifact_paths`: local files downloaded from returned artifacts


In [ ]:
last_result
